<a href="https://colab.research.google.com/github/puderzuckerimschnee-max/bxmid03-hs26-team-2/blob/main/checks/Team2_U-Wert_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# Team 2 · U-Wert der Aussenwände berechnen, ins Modell schreiben, prüfen (Starter)

Euer Soll-Prozess: Die Bauphysik liefert den Wandaufbau, daraus entsteht der U-Wert, das Modell trägt ihn, das IDS prüft ihn.
Dieses Notebook setzt das an einem echten Architekturmodell um:

1. Aussenwände finden (`Pset_WallCommon.IsExternal`)
2. Wandtypen und ihre Schichten auflisten
3. **Aufbautabelle:** Dicke, λ und Quelle pro Schicht. **Die füllt ihr aus.**
4. U = 1 / (Rsi + Σ d/λ + Rse) pro Wandtyp, eine Wand von Hand nachgerechnet
5. Ergebnis pro Wand: bestanden, durchgefallen oder fehlend
6. U-Wert als `Pset_WallCommon.ThermalTransmittance` in eine Kopie des IFC schreiben
7. IDS-Prüfung der Kopie: vorhanden und ≤ Grenzwert

Modell: `models/BIMcollab_Example_ARC.ifc` (BIMcollab-Beispielmodell, IFC2X3, Längen in mm).
Werkzeuge: Python 3.12, IfcOpenShell 0.9.0.post1, ifctester 0.9.0, pandas.

## 1 · Pakete installieren

In [ ]:
%pip install -q ifcopenshell==0.9.0.post1 ifctester==0.9.0 pandas

## 2 · Modell laden
Lokal aus dem Repository, sonst (Colab) von GitHub.

In [ ]:
import os, urllib.request
MODELL = "../models/BIMcollab_Example_ARC.ifc"
URL = "https://raw.githubusercontent.com/puderzuckerimschnee-max/bxmid03-hs26-team-2/main/models/BIMcollab_Example_ARC.ifc"
if not os.path.exists(MODELL):
    MODELL = "BIMcollab_Example_ARC.ifc"
    if not os.path.exists(MODELL):
        urllib.request.urlretrieve(URL, MODELL)

import ifcopenshell, ifcopenshell.util.element as ue, ifcopenshell.util.unit as uu
modell = ifcopenshell.open(MODELL)
skala = uu.calculate_unit_scale(modell)   # Modelleinheit → m
print(f"{MODELL}: {modell.schema}, Länge × {skala} = m, {len(modell.by_type('IfcWall'))} Wände")

## 3 · Aussenwände
Nur Wände mit `IsExternal = TRUE`. Fehlende Werte bleiben leer, nicht 0.

In [ ]:
import pandas as pd

def schichten(w):
    mat = ue.get_material(w, should_skip_usage=True)
    if mat is None:
        return []
    if mat.is_a("IfcMaterialLayerSet"):
        return [l.Material.Name if l.Material else None for l in mat.MaterialLayers]
    if mat.is_a("IfcMaterialList"):
        return [m.Name for m in mat.Materials]
    return [getattr(mat, "Name", None)]

zeilen = []
for w in modell.by_type("IfcWall"):
    p = ue.get_psets(w)
    common = p.get("Pset_WallCommon", {})
    typ = ue.get_type(w)
    breite = p.get("BaseQuantities", {}).get("Width") or p.get("Qto_WallBaseQuantities", {}).get("Width")
    zeilen.append({
        "GlobalId": w.GlobalId,
        "Name": w.Name,
        "Wandtyp": typ.Name if typ else w.Name,
        "IsExternal": common.get("IsExternal"),
        "Dicke_Modell_m": round(breite * skala, 3) if breite else None,
        "Schichten_Modell": " | ".join(str(s) for s in schichten(w)),
        "U_im_Modell": common.get("ThermalTransmittance"),
    })
alle = pd.DataFrame(zeilen)
print(alle["IsExternal"].value_counts(dropna=False).to_string())
aussen = alle[alle["IsExternal"] == True].copy()
print(f"\n{len(aussen)} Aussenwände, davon mit U-Wert im Modell: {aussen['U_im_Modell'].notna().sum()}")

## 4 · Wandtypen der Aussenwände
Pro Typ: Anzahl, Dicke im Modell, Schichten laut Modell. Achtung: Das Modell kennt nur Materialnamen, **keine Schichtdicken und keine λ-Werte**.
Prüft auch, ob jede Wand wirklich eine Aussenwand ist (z. B. Typ «internal wall» mit `IsExternal = TRUE`?).

In [ ]:
typen = (aussen.groupby(["Wandtyp", "Dicke_Modell_m", "Schichten_Modell"], dropna=False)
         .size().rename("Anzahl").reset_index().sort_values("Anzahl", ascending=False))
typen

## 5 · Aufbautabelle: eure Fachleistung

Eine Zeile pro Schicht, von innen nach aussen. Pro Schicht entweder **d und λ** oder direkt **R** (z. B. Luftschicht).
- `Quelle`: woher der Wert kommt (Produktdatenblatt, SIA 279, Lehrbuch, Annahme). **Keine Werte ohne Quelle.**
- Startpunkt: Die Tabelle ist aus den Materialnamen im Modell vorbereitet, d und λ sind leer.
- Speichert sie als `aufbau.csv` neben dem Notebook. Wenn die Datei existiert, wird sie statt der Vorlage gelesen.

In [ ]:
AUFBAU_DATEI = "aufbau.csv"
if os.path.exists(AUFBAU_DATEI):
    aufbau = pd.read_csv(AUFBAU_DATEI, sep=";")
    print(f"{AUFBAU_DATEI} gelesen: {len(aufbau)} Schichten")
else:
    vorlage = []
    for _, t in typen.drop_duplicates("Wandtyp").iterrows():
        for i, s in enumerate(t["Schichten_Modell"].split(" | "), 1):
            vorlage.append({"Wandtyp": t["Wandtyp"], "Nr": i, "Schicht": s,
                            "d_m": None, "lambda_W_mK": None, "R_m2K_W": None, "Quelle": None})
    aufbau = pd.DataFrame(vorlage)
    aufbau.to_csv("aufbau_vorlage.csv", sep=";", index=False, encoding="utf-8-sig")
    print("Vorlage geschrieben: aufbau_vorlage.csv → ausfüllen, als aufbau.csv speichern, Notebook neu starten")
aufbau

## 6 · U-Wert berechnen

U = 1 / (Rsi + Σ R_i + Rse), mit R_i = d/λ oder direkt R.
Wärmeübergangswiderstände für horizontalen Wärmestrom nach EN ISO 6946: Rsi = 0.13, Rse = 0.04 m²K/W.
Zuerst die Funktion an einem Beispiel von Hand prüfen.

In [ ]:
RSI, RSE = 0.13, 0.04

def u_wert(schichten_df):
    """U in W/(m²K) oder None, wenn eine Schicht unvollständig ist."""
    summe_r = 0.0
    for _, s in schichten_df.iterrows():
        if pd.notna(s["R_m2K_W"]):
            summe_r += s["R_m2K_W"]
        elif pd.notna(s["d_m"]) and pd.notna(s["lambda_W_mK"]) and s["lambda_W_mK"] > 0:
            summe_r += s["d_m"] / s["lambda_W_mK"]
        else:
            return None
    return 1 / (RSI + summe_r + RSE)

# Handrechnung: 0.20 m Dämmung mit λ = 0.035 → R = 5.714, U = 1 / (0.13 + 5.714 + 0.04) = 0.170
test = pd.DataFrame([{"d_m": 0.20, "lambda_W_mK": 0.035, "R_m2K_W": None}])
assert abs(u_wert(test) - 0.170) < 0.001
print(f"Handrechnung bestätigt: U = {u_wert(test):.3f} W/(m²K)")

In [ ]:
u_typ = {}
for typ, gruppe in aufbau.groupby("Wandtyp"):
    u = u_wert(gruppe.sort_values("Nr"))
    d = gruppe["d_m"].sum(min_count=len(gruppe))
    u_typ[typ] = {"U_berechnet": None if u is None else round(u, 3), "Dicke_Aufbau_m": d}
u_typ = pd.DataFrame.from_dict(u_typ, orient="index")
u_typ

## 7 · Ergebnis pro Wand

- `GRENZWERT` und `GRENZWERT_QUELLE`: **eine** Quelle, selbst nachgelesen (Norm, Ausgabe, Tabelle).
- Plausibilität: Summe der Schichtdicken soll zur Wanddicke im Modell passen (± 5 mm).

In [ ]:
GRENZWERT = 0.17          # W/(m²K)
GRENZWERT_QUELLE = "TODO: Norm, Ausgabe, Tabelle selbst nachlesen"

erg = aussen.join(u_typ, on="Wandtyp")

def bewerten(z):
    if pd.isna(z.U_berechnet):
        return "fehlend", "Aufbau unvollständig (d, λ oder R fehlt)"
    if pd.notna(z.Dicke_Aufbau_m) and pd.notna(z.Dicke_Modell_m) and abs(z.Dicke_Aufbau_m - z.Dicke_Modell_m) > 0.005:
        hinweis = f"; Aufbau {z.Dicke_Aufbau_m:.3f} m ≠ Modell {z.Dicke_Modell_m:.3f} m"
    else:
        hinweis = ""
    if z.U_berechnet <= GRENZWERT:
        return "bestanden", f"U {z.U_berechnet:.3f} ≤ {GRENZWERT}{hinweis}"
    return "durchgefallen", f"U {z.U_berechnet:.3f} > {GRENZWERT}{hinweis}"

erg[["Ergebnis", "Grund"]] = erg.apply(lambda z: pd.Series(bewerten(z)), axis=1)
print(erg["Ergebnis"].value_counts().to_string())
erg[["GlobalId", "Wandtyp", "U_berechnet", "Ergebnis", "Grund"]].head(10)

## 8 · U-Wert ins Modell schreiben (Kopie)
Nur Wände mit berechnetem U. Das Original bleibt unverändert.

In [ ]:
import ifcopenshell.api
kopie = ifcopenshell.open(MODELL)
geschrieben = 0
for _, z in erg[erg["U_berechnet"].notna()].iterrows():
    w = kopie.by_guid(z.GlobalId)
    pset = ue.get_pset(w, "Pset_WallCommon")
    if pset:
        pset = kopie.by_id(pset["id"])
    else:
        pset = ifcopenshell.api.run("pset.add_pset", kopie, product=w, name="Pset_WallCommon")
    ifcopenshell.api.run("pset.edit_pset", kopie, pset=pset, properties={"ThermalTransmittance": float(z.U_berechnet)})
    geschrieben += 1
AUSGABE_IFC = "BIMcollab_Example_ARC_mit_U.ifc"
kopie.write(AUSGABE_IFC)
print(f"{geschrieben} Wände mit U-Wert → {AUSGABE_IFC}")

## 9 · IDS-Prüfung der Kopie
Regel: Jede Aussenwand hat `Pset_WallCommon.ThermalTransmittance` ≤ Grenzwert. Das IDS wird hier erzeugt und als Datei gespeichert,
damit ihr es auch in IFC-Lite laden könnt.

In [ ]:
import ifctester.ids as ids, ifctester.reporter as reporter

def u_wert_ids():
    regel = ids.Ids(title="Team 2: U-Wert Aussenwände")
    spec = ids.Specification(name="Aussenwand U-Wert vorhanden und ≤ Grenzwert", ifcVersion=["IFC2X3", "IFC4"])
    spec.applicability.append(ids.Entity(name="IFCWALL"))
    spec.applicability.append(ids.Property(propertySet="Pset_WallCommon", baseName="IsExternal",
                                           value="true", dataType="IFCBOOLEAN"))
    spec.requirements.append(ids.Property(propertySet="Pset_WallCommon", baseName="ThermalTransmittance",
                                          dataType="IFCTHERMALTRANSMITTANCEMEASURE", cardinality="required",
                                          value=ids.Restriction(base="double", options={"maxInclusive": GRENZWERT})))
    regel.specifications.append(spec)
    return regel

u_wert_ids().to_xml("Team2_U-Wert_Aussenwand.ids")
for datei in (MODELL, AUSGABE_IFC):
    regel = u_wert_ids()   # pro Modell neu, sonst Fehler in ifctester
    m = ifcopenshell.open(datei)   # Referenz behalten, solange der Bericht läuft
    regel.validate(m)
    s = reporter.Json(regel).report()["specifications"][0]
    print(f"{os.path.basename(datei)}: {s['total_applicable_pass']} von {s['total_applicable']} Aussenwänden bestanden")

## 10 · Export

In [ ]:
erg[["GlobalId", "Name", "Wandtyp", "Dicke_Modell_m", "U_berechnet", "Ergebnis", "Grund"]] \
   .assign(Soll=f"≤ {GRENZWERT} W/(m²K) ({GRENZWERT_QUELLE})") \
   .to_csv("u_wert_aussenwaende.csv", sep=";", index=False, encoding="utf-8-sig")
print("geschrieben: u_wert_aussenwaende.csv")
try:
    from google.colab import files
    for f in ("u_wert_aussenwaende.csv", "aufbau_vorlage.csv", "Team2_U-Wert_Aussenwand.ids"):
        if os.path.exists(f):
            files.download(f)
except ImportError:
    pass

## 11 · Fragen für euer MVP
- Welche Wandtypen sind wirklich Aussenwände? Stimmt `IsExternal` im Modell?
- Woher kommen eure λ-Werte? Eine Quelle pro Material.
- Eine Wand von Hand nachrechnen und im README zeigen.
- Negativfall: ein Aufbau mit dünnerer Dämmung → U über dem Grenzwert → durchgefallen, gleiche Regel.
- Was macht der Architekt mit einer durchgefallenen Wand (BCF-Rückmeldung)?